# pickett: ทำนาย ฟิต และวิเคราะห์สเปกตรัมใน Jupyter

notebook นี้ใช้ไฟล์ตัวอย่างใน `test/golden/` ของ repo และทำงานทั้งหมดในหน่วยความจำ ไม่สร้างไฟล์ใด ๆ

1. ทำนายสเปกตรัมของ CH₃CN (SPCAT)
2. ดู partition function
3. ฟิตค่าคงที่ของโมเลกุลแบบ SO₂ (SPFIT)
4. วิเคราะห์สเปกตรัมจำลอง: ลบ baseline, หาพีค, ฟิตเส้น, assign และฟิตกลับ
5. รันหลายงานพร้อมกัน

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import pickett
from pickett import spectrum as sp

# โฟลเดอร์ test/golden ของ repo (แก้ path ถ้าเปิด notebook จากที่อื่น)
G = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "test" / "golden").is_dir()) / "test" / "golden"
print("thread-safe C library:", pickett.thread_safe())

## 1. ทำนายสเปกตรัม (SPCAT)

`predict(var, int)` รับ path หรือข้อความของไฟล์ `.var` และ `.int` ค่าใน `lines` เป็นค่าความละเอียดเต็ม ไม่ได้ถูกปัดเหมือนในไฟล์ `.cat`

In [ ]:
ch3cn = pickett.predict(G / "spcat_prolate/input/prolate.var", G / "spcat_prolate/input/prolate.int")
print(ch3cn)
ch3cn.lines_frame().sort_values("lgint", ascending=False).head()

In [ ]:
l = ch3cn.lines
fig, ax = plt.subplots(figsize=(9, 3))
ax.vlines(l["freq"] / 1e3, -9, l["lgint"], lw=0.6)
ax.set(xlabel="frequency (GHz)", ylabel="log$_{10}$ intensity", ylim=(-9, -1), title="CH$_3$CN, 300 K")
plt.show()

## 2. Partition function 1–1000 K

In [ ]:
t, q = ch3cn.partition[:, 0], ch3cn.partition[:, 1]
fig, ax = plt.subplots(figsize=(5, 3))
ax.loglog(t, q)
ax.set(xlabel="T (K)", ylabel="Q(T)")
plt.show()
print("Q(300 K) =", ch3cn.partition_at(300.0), "  Q(2.725 K) =", ch3cn.partition_at(2.725))

## 3. ฟิตค่าคงที่ (SPFIT)

เริ่มจากค่าคงที่ที่คลาดเคลื่อน (`asym.par`) และเส้นที่ assign แล้ว 50 เส้น (`asym.lin`)

In [ ]:
result = pickett.fit(G / "spfit_asym/input/asym.par", G / "spfit_asym/input/asym.lin")
print(result)
result.params_frame()

In [ ]:
# นำ .var ที่ได้ไปทำนายต่อทันที
so2 = pickett.predict(result.var, G / "spcat_asym/input/asym.int")
so2.lines_frame().head(3)

## 4. วิเคราะห์สเปกตรัม (สเปกตรัมจำลอง)

สร้าง "สเปกตรัมที่วัดได้" จากค่าคงที่จริง โดยใส่ baseline ที่เป็นคลื่นและ noise ลงไป จากนั้นเริ่มจากค่าคงที่ที่คลาดเคลื่อน (A ผิดไป 8.5 MHz) แล้วทำ: ลบ baseline → หาพีค → ฟิตเส้น → assign → SPFIT วนหลายรอบ

In [ ]:
truth = pickett.predict(G / "spcat_asym/input/asym.var", G / "spcat_asym/input/asym.int")
f, y = sp.catalog_spectrum(truth, 100_000.0, 250_000.0, 0.2, "gaussian", 1.0)
rng = np.random.default_rng(1)
y = y + 4e-6 + 2e-6 * np.sin(f / 9000.0) + 2e-7 * rng.standard_normal(len(f))

base = sp.asls(y, lam=1e8, p=0.002, iterations=15)
flat = y - base
peaks = sp.find_peaks(f, flat, min_snr=8, min_distance=3)
fits = sp.fit_peaks(f, flat, peaks)
ok = fits["converged"]
centres, errs = fits["center"][ok], np.maximum(fits["center_err"][ok], 1e-4)
print(f"{len(f)} points, noise {sp.noise_level(flat):.2e}, {len(peaks['freq'])} peaks, {ok.sum()} line fits")

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(f / 1e3, y, lw=0.3, label="measured")
ax.plot(f / 1e3, base, lw=1, label="AsLS baseline")
ax.set(xlabel="frequency (GHz)", ylabel="intensity", ylim=(0, 2e-5),
       title="y axis zoomed to show the baseline and noise")
ax.legend()
plt.show()

In [ ]:
par = (G / "spfit_asym/input/asym.par").read_text()
int_ = G / "spcat_asym/input/asym.int"
for tol, min_lgint, floor in [(60, -4, 3.0), (10, -5.5, 0.5), (2, -7, 0.05), (0.5, -7, 0.0)]:
    pred = pickett.predict(par, int_)
    m = sp.match_lines(centres, pred, tolerance=tol, min_lgint=min_lgint)
    lin = sp.to_lin(pred, m, np.maximum(errs[m["observed"]], floor))
    fitres = pickett.fit(par, lin)
    par = fitres.par
    print(f"tol {tol:5}: {len(m['obs']):3d} assigned, A = {fitres.value('A'):.5f}, weighted RMS {fitres.rms_error:.3f}")
print("true A = 60778.55")
fitres.params_frame()

In [ ]:
# measured vs fitted line in a small window
k = int(np.argmax(fits["amplitude"][ok]))
c = centres[k]
w = (f > c - 4) & (f < c + 4)
model = sp.synthesize([fits["center"][ok][k]], [fits["amplitude"][ok][k]], f[w], "gaussian", fits["fwhm"][ok][k])
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(f[w] - c, flat[w], ".", ms=3, label="measured")
ax.plot(f[w] - c, model, label="fit")
ax.set(xlabel=f"frequency - {c:.4f} MHz", ylabel="intensity")
ax.legend()
plt.show()

## 5. รันหลายงานพร้อมกัน

`predict_many` กระจายงานไปทุก core ตัวอย่างนี้สแกนค่า A ±60 MHz แล้วดูว่าความถี่ของเส้นหนึ่งเปลี่ยนไปอย่างไร

In [ ]:
var_text = (G / "spcat_asym/input/asym.var").read_text()
int_text = int_.read_text()
offsets = np.linspace(-60, 60, 13)
def with_a(delta):
    out = []
    for line in var_text.splitlines():
        if line.strip().endswith("/A"):
            v = line.split()[1]
            line = line.replace(v, f"{float(v) + delta:.15E}")
        out.append(line)
    return chr(10).join(out) + chr(10)
scan = pickett.predict_many([(with_a(d), int_text) for d in offsets])
f0 = [p.lines["freq"][p.lines["lgint"].argmax()] for p in scan]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(offsets, np.array(f0) - f0[6], "o-")
ax.set(xlabel="ΔA (MHz)", ylabel="Δν of the strongest line (MHz)")
plt.show()